# Mindbaton brain 2B: train it on Kaggle's free GPU

Fine-tunes **Qwen3.5-2B** (Apache-2.0) to read chat messages into facts, the way Mindbaton's rules do, but for any
phrasing. Everything it learns from is made up (templates and examples written by an open model); none of it is anyone's
real messages. It then scores itself on Mindbaton's test sets, converts to GGUF (~1.3 GB, what llama.cpp runs) and
uploads the model to your Hugging Face account.

**Before you press Run All** (right-hand panel, *Session options*):
1. **Accelerator: GPU T4 x2** (a P100 is too old for this).
2. **Internet: on** (needs a phone-verified Kaggle account).
3. **Add-ons → Secrets → Add secret**: name `HF_TOKEN`, value a Hugging Face token with *Write* access
   (huggingface.co → Settings → Access Tokens). Tick it for this notebook. Without it, the model stays in the
   notebook's *Output* tab instead.

Then **Run All**. About 1–1.5 hours; the last cell prints the scores and the link to your model.

In [ ]:
BRANCH = "brain-upgrade"          # the Mindbaton branch with the training data builder (main once it's merged)
BASE = "Qwen/Qwen3.5-2B"
REPO_NAME = "mindbaton-brain-2b"   # on your Hugging Face account
MAX_LEN = 512                      # chat messages are short
%pip install -q unsloth

In [ ]:
# 1. The lessons: Mindbaton builds them itself from its own templates and examples (no one's real messages)
!rm -rf /kaggle/working/mindbaton && git clone -q --depth 1 -b {BRANCH} https://github.com/DkshByte/mindbaton /kaggle/working/mindbaton
!cd /kaggle/working/mindbaton && python3 learn.py --dataset /kaggle/working/data.jsonl

In [ ]:
# 2. The model, with small trainable adapters (LoRA)
from unsloth import FastLanguageModel
import torch, json, os, sys
model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=MAX_LEN, load_in_4bit=False,
                                                     load_in_16bit=True, full_finetuning=False,
                                                     dtype=torch.float16)  # a T4 has no bfloat16
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                                         use_gradient_checkpointing="unsloth", random_state=3407, max_seq_length=MAX_LEN)

def chat(messages, **kw):
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False, **kw)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, **kw)

In [ ]:
# 3. Train: one pass over ~20k lessons, learning only the answers
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
ds = load_dataset("json", data_files="/kaggle/working/data.jsonl", split="train")
ds = ds.map(lambda b: {"text": [chat(m) for m in b["messages"]]}, batched=True, remove_columns=ds.column_names)
assert "<|im_start|>assistant\n" in ds[0]["text"], "this model formats chats differently: fix the markers below\n" + ds[0]["text"][:600]
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=SFTConfig(
    dataset_text_field="text", max_seq_length=MAX_LEN, per_device_train_batch_size=8, gradient_accumulation_steps=2,
    warmup_steps=20, num_train_epochs=1, learning_rate=2e-4, logging_steps=25, optim="adamw_8bit", weight_decay=0.01,
    lr_scheduler_type="linear", fp16=True, bf16=False, seed=3407, output_dir="/kaggle/working/outputs", report_to="none", dataset_num_proc=2))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

In [ ]:
# 4. Score it on Mindbaton's test sets (none of these sentences were in the lessons)
sys.path.insert(0, "/kaggle/working/mindbaton")
os.environ["MINDBATON_DATA"] = "/tmp/mb"
import bench, brain, learn
FastLanguageModel.for_inference(model)

def read(text):
    prompt = chat([{"role": "system", "content": learn.SYSTEM}, {"role": "user", "content": text}], add_generation_prompt=True)
    ids = tokenizer(text=prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    ans = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        return json.loads(ans[ans.find("{"):ans.rfind("}") + 1])["facts"]
    except Exception:
        return []

same = lambda a, b: brain.key(a) == b or brain.same_thing(brain.key(a), b)
def score(facts, none, gone):
    found = sum(any(f.get("op") == "+" and f.get("rel") == rel and same(f.get("what", ""), k) for f in read(t)) for t, (rel, k) in facts)
    false = sum(any(f.get("who") == "me" and f.get("op") == "+" and f.get("rel") not in brain.KIN for f in read(t)) for t in none)
    back = sum(any(f.get("op") == "-" and same(f.get("what", ""), k) for f in read(t)) for t, k in gone)
    return {"facts": f"{found}/{len(facts)}", "no false facts": f"{len(none) - false}/{len(none)}", "taken back": f"{back}/{len(gone)}"}

results = {"everyday phrasings": score([(t, w[1:]) for t, w in bench.NATURAL], bench.NATURAL_NOT, bench.NATURAL_GONE),
           "fresh set 2": score(bench.NATURAL2["facts"], bench.NATURAL2["none"], bench.NATURAL2["gone"]),
           "fresh set 3": score(bench.NATURAL3["facts"], bench.NATURAL3["none"], bench.NATURAL3["gone"])}
print(json.dumps(results, indent=1))
for t in ["5-a-side football every thursday after work", "zero caffeine since march", "my dad plays tennis every sunday",
          "what's the best laptop for coding?"]:
    print(t, "->", read(t))

In [ ]:
# 5. GGUF for llama.cpp (4-bit, ~1.3 GB) and the adapters, then upload to your Hugging Face account
out = "/kaggle/working/model"
model.save_pretrained_gguf(out, tokenizer, quantization_method="q4_k_m")
model.save_pretrained(out + "/lora"); tokenizer.save_pretrained(out + "/lora")
json.dump(results, open(out + "/results.json", "w"), indent=1)
open(out + "/README.md", "w").write(f"""---
license: apache-2.0
base_model: {BASE}
tags: [mindbaton, memory, gguf, information-extraction]
---
# Mindbaton brain 2B

{BASE} fine-tuned (LoRA) to read one chat message into the lasting facts it states about the writer or the people they
name, as JSON. It runs in the background in [Mindbaton](https://github.com/DkshByte/mindbaton) through llama.cpp;
Mindbaton's rules check every fact before it is saved. Trained only on made-up examples (Mindbaton's templates and
chat messages written by gpt-oss-120b), never on anyone's messages.

Scores on Mindbaton's test sets (sentences it never saw): `{json.dumps(results)}`
""")
from huggingface_hub import HfApi
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
if token:
    api = HfApi(token=token)
    repo = f"{api.whoami()['name']}/{REPO_NAME}"
    api.create_repo(repo, exist_ok=True)
    ggufs = [f for f in os.listdir(out) if f.lower().endswith(".gguf") and "q4_k_m" in f.lower()]
    api.upload_folder(folder_path=out, repo_id=repo, allow_patterns=ggufs + ["README.md", "results.json", "lora/*"])
    print("Done: https://huggingface.co/" + repo, ggufs)
else:
    print("No HF_TOKEN secret: the model is in the Output tab,", os.listdir(out))